In [1]:
import torch
from architecture import PINN_base

In [2]:
d = 4
D = d+1
layers = 5*[64]
model = PINN_base(D,layers,1)

In [3]:
def loss(u):
    return torch.mean(u)

In [4]:
bs = 10
X = torch.rand((bs, D))
X[:,:-1] *= 2.0
X[:,:-1] -= 1.0
X[:,-1] *= 5.0
X[0]

tensor([ 0.6405,  0.1481,  0.1138, -0.2745,  3.8166])

In [5]:
t = X[:,-1]
t

tensor([3.8166, 2.2636, 4.7755, 2.6523, 4.2117, 4.7391, 4.3225, 2.0210, 1.0511,
        3.9525])

In [6]:
# compute loss for each time window
# then combine

In [7]:
T = 5.0
M = 4
t0 = 0.0
tM = T
t_discr = torch.linspace(t0, tM, M)
t_discr

tensor([0.0000, 1.6667, 3.3333, 5.0000])

In [8]:
# eval the residual
u = model(X)
res = u # + u_grad * v + u_laplace
res.shape

torch.Size([10, 1])

In [9]:
t_discr

tensor([0.0000, 1.6667, 3.3333, 5.0000])

In [10]:
# want interval [,)
bin_indx = torch.bucketize(t, t_discr[1:-1], right=True)
bin_indx

/tmp/ipykernel_3970/1596093841.py:2: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  bin_indx = torch.bucketize(t, t_discr[1:-1], right=True)


tensor([2, 1, 2, 1, 2, 2, 2, 1, 0, 2])

In [11]:
outputs = []
losses = []
for i in range(len(t_discr) - 1):
    mask = (bin_indx == i)
    outputs.append(res[mask])
    losses.append(loss(res[mask]).item())
print(losses)
outputs

[0.23365375399589539, 0.22209776937961578, 0.20065003633499146]


[tensor([[0.2337]], grad_fn=<IndexBackward0>),
 tensor([[0.2266],
         [0.2116],
         [0.2281]], grad_fn=<IndexBackward0>),
 tensor([[0.2035],
         [0.1945],
         [0.2011],
         [0.1995],
         [0.2053],
         [0.2000]], grad_fn=<IndexBackward0>)]

In [12]:
indices = t.sort().indices
res[indices]

tensor([[0.2337],
        [0.2281],
        [0.2266],
        [0.2116],
        [0.2035],
        [0.2000],
        [0.2011],
        [0.2053],
        [0.1995],
        [0.1945]], grad_fn=<IndexBackward0>)

In [13]:
# full loss
outputs = []
losses = [] # save losses for debugg?
eps = 1.0
loss_cum = torch.tensor(0.0)
loss_tot = torch.tensor(0.0)
w_ms = [] # save causal weights for debugg
for i in range(len(t_discr) - 1):
    mask = (bin_indx == i)
    loss_val = loss(res[mask])
    w = torch.exp(-eps*loss_cum)
    loss_tot += w * loss_val
    loss_cum += loss_val
    w_ms.append(w)
    outputs.append(res[mask])
    losses.append(loss_val.item())

print(losses)
print(loss_tot)
print(w_ms)
outputs

[0.23365375399589539, 0.22209776937961578, 0.20065003633499146]
tensor(0.5367, grad_fn=<AddBackward0>)
[tensor(1.), tensor(0.7916, grad_fn=<ExpBackward0>), tensor(0.6340, grad_fn=<ExpBackward0>)]


[tensor([[0.2337]], grad_fn=<IndexBackward0>),
 tensor([[0.2266],
         [0.2116],
         [0.2281]], grad_fn=<IndexBackward0>),
 tensor([[0.2035],
         [0.1945],
         [0.2011],
         [0.1995],
         [0.2053],
         [0.2000]], grad_fn=<IndexBackward0>)]